# 90 — ColBERT DEV experiments (clean harness)

Single, clean DEV harness for **all ColBERT changes**, replacing the nb82 tangle. One CONFIG
block drives every build + eval so train / index / dev-eval-pack can never desync (the review's
C1/C2/C3). Reuses the tested shared helpers (`build_retrieval_query(mode='compact_colbert')`,
`make_colbert_doc_text_fn`, `resolve_sub_queries`) so DEV measures exactly what serve ships.

**Experiments covered**
- `baseline`        — current ColBERT (full-dialog query, raw-metadata docs).
- `exp216_compact`  — compact query (goal+culture+last user turn), q_len 160.   (query-side)
- `exp217_tags`     — curated-tag enriched docs, d_len 128.                       (doc-side)
- `exp216_217`      — both.

**Binding gate (pre-registered):** TURN-1 **wall** recall@20/@100 (the only metric that clears
the 80-session ±0.05 noise) + does the variant raise **union** recall@100 (marginal, not
standalone — ColBERT is 1 of ~7 channels and BM25 already matches raw tags). nDCG guard for
MaxSim score-inflation. Internal/standalone val is a TRAP — do NOT gate on it.

In [ ]:
# 0) Setup (nb82 convention). Disable JAX GPU preallocation BEFORE any import pulls JAX
# in; mount Drive; clone the branch; symlink the persistent Drive caches; install deps.
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'recall-union-lgbm'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

# Symlink the persistent caches from Drive (retrieval_v2 = sasrec/colbert/lgbm/...; dense = Qwen).
DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, drive_subdir in [('retrieval_v2', 'recsys2026_retrieval_v2_cache'),
                           ('dense', 'recsys2026_dense_cache')]:
    src = f'{DRIVE_BASE}/{drive_subdir}'; dst = f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

!pip install -q --upgrade 'transformers>=4.40' 'accelerate>=0.30' 'peft>=0.11' \
    'datasets' 'pandas<3.0' 'tqdm' 'huggingface_hub' 'sentence-transformers>=3.0' \
    'FlagEmbedding>=1.3' 'bm25s' 'lightgbm' 'scikit-learn' 'omegaconf' 'pyyaml' 'pylate>=1.1.0'

In [ ]:
# 0b) Version guard + constants. Proves this runtime is on the LATEST pushed code (fresh
# clone AND fresh imports — a live kernel caches the OLD module). If this fails: Runtime ->
# Restart session, re-run cell 0, then this cell.
import sys, subprocess
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
sha = subprocess.check_output(['git', '-C', '/content/recsys2026', 'rev-parse', '--short', 'HEAD']).decode().strip()
print('repo HEAD =', sha, '(EXP-216/217 = f7202b5 or later)')
from mcrs.retrieval_modules.colbert_late import make_colbert_doc_text_fn, build_tag_vocab
from mcrs.crs_baseline import build_retrieval_query  # noqa: F401 (proves compact_colbert present)
assert build_tag_vocab([['jazz', 'jazz']], min_freq=1) == {'jazz': 1}, \
    'STALE CODE: EXP-217 doc-frequency vocab missing — Restart session + re-run cell 0'
print('OK: EXP-216/217 helpers loaded (make_colbert_doc_text_fn + doc-freq vocab). Safe to proceed.')

ITEM_DB  = 'talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
CORPUS   = ['track_name', 'artist_name', 'album_name']
CACHE_DIR = '/content/recsys2026/experiments/cache'   # the symlinked Drive cache
print('CACHE_DIR =', CACHE_DIR)

In [ ]:
# 1) CONFIG — the single source of truth. Build/eval flags derive from HERE, so train,
#    index, and the dev-eval pack always use the SAME doc/query recipe (C3: no desync).
TAG_MIN_FREQ, TAG_TOP_K = 50, 15   # EXP-217 curation (measured: vocab~3.3k, enriched p99 92 tok)

VARIANTS = {
    'baseline':       dict(compact_query=False, enrich_tags=False, q_len=96,  d_len=96,
                           model='music-colbert-v1',              index='colbert-music-v1'),
    'exp216_compact': dict(compact_query=True,  enrich_tags=False, q_len=160, d_len=96,
                           model='music-colbert-compact-v1',      index='colbert-music-compact-v1'),
    'exp217_tags':    dict(compact_query=False, enrich_tags=True,  q_len=96,  d_len=128,
                           model='music-colbert-tagenriched-v1',  index='colbert-music-tagenriched-v1'),
    'exp216_217':     dict(compact_query=True,  enrich_tags=True,  q_len=160, d_len=128,
                           model='music-colbert-compact-tags-v1', index='colbert-music-compact-tags-v1'),
}
COLBERT_DIR = f'{CACHE_DIR}/retrieval_v2/colbert'
def model_path(v): return f"{COLBERT_DIR}/{VARIANTS[v]['model']}"
def index_folder():  return f"{COLBERT_DIR}/plaid"
for k, v in VARIANTS.items():
    print(f"{k:16s} compact={v['compact_query']} enrich={v['enrich_tags']} "
          f"q_len={v['q_len']} d_len={v['d_len']} -> {v['index']}")

## Section 1 — DEV eval set (nb74/nb82 parity) + compact-query variant + wall instrument

In [ ]:
# 2) Build the FULL dev eval set (verbatim nb74/nb82 cell-5) AND the compact query variant.
import numpy as np, pandas as pd
from datasets import load_dataset
from mcrs.db_item.music_catalog import MusicCatalogDB
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.sasrec_model import build_user_dialog
from mcrs.crs_baseline import build_retrieval_query

item_db = MusicCatalogDB(ITEM_DB, ['all_tracks'], CORPUS)
dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')

queries, queries_compact, golds, user_ids, played, user_dialogs = [], [], [], [], [], []
turn_numbers = []
for sess in dev:
    df = pd.DataFrame(sess['conversations'])
    goal = sess.get('conversation_goal') or {}
    goal_txt = (goal.get('listener_goal') or '').strip()
    up = sess.get('user_profile')
    for _, music in df[df['role'] == 'music'].iterrows():
        tn = int(music['turn_number'])
        prior = df[(df['turn_number'] < tn) | ((df['turn_number'] == tn) & (df['role'] == 'user'))]
        # full-dialog query (raw/baseline) — music turns rendered as metadata
        prior_turns = [{'role': ('assistant' if t['role'] == 'music' else t['role']),
                        'content': (item_db.id_to_metadata(t['content']) if t['role'] == 'music'
                                    else t['content'])} for _, t in prior.iterrows()]
        _q = chr(10).join(f"{t['role']}: {t['content']}" for t in prior_turns)
        if goal_txt:
            _q = _q + chr(10) + 'goal: ' + goal_txt
        queries.append(_q)
        # compact query (EXP-216) — SAME shared builder as serve/train
        queries_compact.append(build_retrieval_query(
            prior_turns, mode='compact_colbert', goal_text=goal_txt, user_profile=up))
        user_dialogs.append(build_user_dialog(prior.to_dict('records')))
        golds.append(music['content'])
        user_ids.append(sess.get('user_id'))
        played.append(list(df[(df['role'] == 'music') & (df['turn_number'] < tn)]['content']))
        turn_numbers.append(tn)
ctx = [{'history_tids': p, 'user_dialog': ud} for p, ud in zip(played, user_dialogs)]
print('[dev] built', len(queries), 'turns |', sum(t == 1 for t in turn_numbers), 'turn-1')

def recall_at(cands, k, golds=golds):
    return float(np.mean([1.0 if g in c[:k] else 0.0 for c, g in zip(cands, golds)]))

# config-200 union pool `cs` (the baseline the ColBERT rescue is measured against)
sas = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                            CACHE_DIR, extra_config={'use_sasrec': True, 'w_sasrec': 1.0})
cs = sas.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
print('[union+SASRec] recall@20=%.4f @100=%.4f' % (recall_at(cs, 20), recall_at(cs, 100)))

In [ ]:
# 3) Turn-1 / WALL recall instrument (nb82 cell-6). TURN-1 wall recall is THE binding gate.
from mcrs.eval_ndcg import recall_by_turn

def _artist_of(tid):
    md_ = item_db.metadata_dict.get(tid) or {}
    a = md_.get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

is_wall = np.array([_artist_of(g) not in {_artist_of(t) for t in p} for g, p in zip(golds, played)])
turn1 = np.array([t == 1 for t in turn_numbers])
cold = turn1; warm = ~turn1
t1_idx = [i for i, t in enumerate(turn_numbers) if t == 1]
print(f'[strat] turns={len(golds)} turn1={int(turn1.sum())} wall={int(is_wall.sum())} '
      f'turn1&wall={int((turn1 & is_wall).sum())}')

def rec_mask(cands, k, mask, golds=golds):
    idx = np.where(mask)[0]
    if len(idx) == 0: return float('nan')
    return float(np.mean([1.0 if golds[i] in cands[i][:k] else 0.0 for i in idx]))

## Section 1.5 — FUSION SWEEP (cheap re-fusion): test "too many channels" + "wrong weights"

Before spending GPU on `exp216_217`, test whether the Blind flatness is a FUSION problem, not a
channel problem. Runs each sub-retriever ONCE, then RE-FUSES under many configs via the pure
`fuse_per_sub`/`fuse_per_sub_quota` (no re-retrieval). Binding metric = turn-1 WALL recall.
Reads: leave-one-out (a channel whose removal doesn't hurt = noise), weight variants (memory:
sasrec underweighted / dense overweighted), k sweep, and EXP-012 channel-quota (currently OFF in
prod). Gate on turn-1 wall recall — NOT internal val. Don't chase the DEV-optimal point; take a
few principled moves to Blind.

In [ ]:
# 4b) FUSION SWEEP over COLD (turn-1) + WARM (turn>=2) in ONE pass. Per segment: run subs
# ONCE, then re-fuse cheaply. pg DROPPED (was inert on cold + Gemini-costly on the ~7k warm set)
# -> $0, 6 channels, apples-to-apples. k-sweep/quota were FLAT on cold -> omitted for focus.
# Reads: COLD confirmed clap_text=NOISE; WARM tells us if clap_text is also noise there and
# whether same_artist (the session lever) finally contributes. Requires Section 1 globals.
import numpy as np
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.rrf import RRF_MODEL

SWEEP_CHANNELS = dict(use_sasrec=True, w_sasrec=1.0, use_colbert=True, w_colbert=1.0,
                      colbert_index_name='colbert-music-v1', use_clap_text=True, w_clap_text=1.0)
union = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                              extra_config=SWEEP_CHANNELS)
F = RRF_MODEL.fuse_per_sub

def run_sweep(idx, seg):
    if not idx:
        print(f'[{seg}] no rows -> skip'); return
    q_s, ctx_s = [queries[i] for i in idx], [ctx[i] for i in idx]
    uid_s, gd = [user_ids[i] for i in idx], [golds[i] for i in idx]
    wl = np.array([bool(is_wall[i]) for i in idx])
    per_sub, labels = union.batch_per_sub_rankings(q_s, user_ids=uid_s, batch_context=ctx_s)
    bw = [s['weight'] for s in union.subs]
    def rec(c, k, m=None):
        ii = list(range(len(gd))) if m is None else list(np.where(m)[0])
        return float(np.mean([1.0 if gd[i] in c[i][:k] else 0.0 for i in ii])) if ii else float('nan')
    def rep(name, c):
        print(f"  {name:30s} r@20={rec(c,20):.4f} r@100={rec(c,100):.4f} wall@100={rec(c,100,wl):.4f}")
    print(); print(f'############ {seg}  (n={len(idx)}, wall={int(wl.sum())}, channels={labels}) ############')
    print('--- baseline + LEAVE-ONE-OUT (>= baseline => that channel is NOISE) ---')
    rep('baseline', F(per_sub, bw, 60, 100))
    for j, lab in enumerate(labels):
        w = list(bw); w[j] = 0.0
        rep(f'  drop {lab}', F(per_sub, w, 60, 100))
    print('--- WEIGHT VARIANTS ---')
    for name, ov in {'sasrec x2': {'sasrec_seq': 2.0}, 'dense ->0.3': {'dense_metadata_qwen3_instruct': 0.3},
                     'colbert x1.5': {'colbert_index': 1.5}}.items():
        rep(name, F(per_sub, [ov.get(l, bw[j]) for j, l in enumerate(labels)], 60, 100))

WARM_MAX = 1500   # CLAP OOM'd encoding all ~7k warm queries -> stride-subsample (SE~0.013, enough)
warm_all = [i for i, t in enumerate(turn_numbers) if t >= 2]
warm_idx = warm_all[::max(1, len(warm_all) // WARM_MAX)]
run_sweep([i for i, t in enumerate(turn_numbers) if t == 1], 'COLD (turn-1, all-wall)')
run_sweep(warm_idx, f'WARM (turn>=2, subsampled {len(warm_idx)}/{len(warm_all)})')

## Section 2 — Build artifacts for ONE variant (GPU, one-time)

Set `VAR` to the variant you are building, then run 4–7 in order. Every flag derives from the
CONFIG row, so train-data / dev-pack / finetune / index are guaranteed consistent. **C1**: the
dev-eval pack is built with the SAME `make_colbert_doc_text_fn` recipe as training+index.
**C2**: `d_len` is threaded into finetune AND index. **Never overwrite the 0.50 `baseline`.**

In [ ]:
# 4) Pick the variant to BUILD, then build the fine-tune triples (TRAIN split).
VAR = 'exp217_tags'   # <- set me: 'exp216_compact' | 'exp217_tags' | 'exp216_217'
V = VARIANTS[VAR]
assert VAR != 'baseline', 'baseline is the existing 0.50 model — do not rebuild/overwrite it'
TRAIN_JSONL = f'{CACHE_DIR}/retrieval_v2/colbert_train_{VAR}.jsonl'
flags = (f"{'--compact-query ' if V['compact_query'] else ''}"
         f"{'--enrich-tags ' if V['enrich_tags'] else ''}"
         f"--tag-min-freq {TAG_MIN_FREQ} --tag-top-k {TAG_TOP_K}")
!python -u scripts/build_colbert_train_data.py --output {TRAIN_JSONL} --cache-dir {CACHE_DIR} \
    --pool-size 100 --k-negs 15 --max-rows 0 {flags}
print('[build-data] DOC RECIPE printed above MUST match the index build (cell 7).')

In [ ]:
# 5) C1 FIX: dev-eval pack built via the SHARED factory -> model selection optimizes the
#    EXACT enriched-doc gate metric (not bare docs). Requires t1_idx, golds, is_wall, cs.
import pickle
from mcrs.retrieval_modules.colbert_late import make_colbert_doc_text_fn
doc_fn, doc_recipe = make_colbert_doc_text_fn(
    item_db, enrich_tags=V['enrich_tags'], tag_min_freq=TAG_MIN_FREQ, tag_top_k=TAG_TOP_K)
q_sel = (queries_compact if V['compact_query'] else queries)
q_t1   = [q_sel[i] for i in t1_idx]
pools_t1 = [cs[i] for i in t1_idx]
golds_t1 = [golds[i] for i in t1_idx]
wall_t1  = [bool(is_wall[i]) for i in t1_idx]
need = sorted({t for pool in pools_t1 for t in pool})
pack = {'queries': q_t1, 'pools': pools_t1, 'golds': golds_t1, 'wall': wall_t1,
        'tid_to_text': {t: doc_fn(t) for t in need}, 'doc_recipe': doc_recipe}
DEV_PACK = f'{CACHE_DIR}/retrieval_v2/colbert_dev_eval_{VAR}.pkl'
pickle.dump(pack, open(DEV_PACK, 'wb'))
print(f'[dev-pack] {len(q_t1)} turn-1 q, {len(need)} docs | recipe={doc_recipe} -> {DEV_PACK}')

In [ ]:
# 6) C2 FIX: fine-tune with q_len AND d_len from the CONFIG row (not the script defaults).
OUT = model_path(VAR)
# Same recipe as nb82: warm-start colbertv2.0, --epochs 2 (1-3; warm-start needs few),
# best checkpoint selected by DEV turn-1 recall@20 every --eval-steps (NOT final epoch).
!python -u scripts/train_colbert.py --train-jsonl {TRAIN_JSONL} \
    --out-dir {OUT} --q-len {V['q_len']} --d-len {V['d_len']} \
    --dev-eval-pack {DEV_PACK} --epochs 2 --batch-size 32 --eval-steps 500 --dev-subset 300
print(f'[finetune] {VAR}: q_len={V["q_len"]} d_len={V["d_len"]} epochs=2 -> {OUT}')

In [ ]:
# 7) C2 FIX: build the PLAID index with --enrich-tags + --d-len matching the model.
#    DOC RECIPE printed here MUST equal cell 4's. New index name -> never clobbers baseline.
ef = (f"{'--enrich-tags ' if V['enrich_tags'] else ''}"
      f"--tag-min-freq {TAG_MIN_FREQ} --tag-top-k {TAG_TOP_K}")
!python -u scripts/build_colbert_index.py --model-dir {OUT} \
    --index-folder {index_folder()} --index-name {V['index']} --d-len {V['d_len']} {ef}
print(f"[build-index] {V['index']} @ d_len={V['d_len']} — verify DOC RECIPE == cell 4.")

## Section 3 — Recall diagnostics + pre-registered gate (per built variant)

In [ ]:
# 8) Per-variant ColBERT full-catalog recall + the wall rescue gate (nb82 cell-17 thesis).
#    Loops over variants whose index exists. baseline uses full-dialog query; compact uses
#    queries_compact (the honest serve-aligned query). GATE = turn-1 wall rescue vs union.
import os
from mcrs.retrieval_modules.colbert_late import ColbertIndexRetriever

EVAL = [v for v in VARIANTS if os.path.exists(f"{index_folder()}/{VARIANTS[v]['index']}")]
print('evaluating built variants:', EVAL)
wall_t1_arr = np.array([bool(is_wall[i]) for i in t1_idx])
cs_t1 = [cs[i] for i in t1_idx]; golds_t1 = [golds[i] for i in t1_idx]
cs_w100 = rec_mask(cs, 100, turn1 & is_wall)
rows = []
colbert_t1 = {}
for v in EVAL:
    V = VARIANTS[v]
    q_t1 = [(queries_compact if V['compact_query'] else queries)[i] for i in t1_idx]
    retr = ColbertIndexRetriever(index_folder=index_folder(), index_name=V['index'],
                                 model_name=model_path(v), q_len=V['q_len'])
    cb = retr.batch_text_to_item_retrieval(q_t1, topk=100)  # full 47k catalog, turn-1
    colbert_t1[v] = cb
    cb_w = float(np.mean([1.0 if golds_t1[j] in cb[j][:100] else 0.0
                          for j in np.where(wall_t1_arr)[0]]))
    comb = [list(dict.fromkeys(list(cs_t1[j][:100]) + list(cb[j][:100]))) for j in range(len(cb))]
    comb_w = float(np.mean([1.0 if golds_t1[j] in comb[j] else 0.0
                            for j in np.where(wall_t1_arr)[0]]))
    missed = [j for j in np.where(wall_t1_arr)[0] if golds_t1[j] not in cs_t1[j][:100]]
    rescued = [j for j in missed if golds_t1[j] in cb[j][:100]]
    rows.append((v, cb_w, comb_w, comb_w - cs_w100, len(rescued), len(missed)))
print(f'\nturn-1&wall recall@100 — union baseline cs = {cs_w100:.4f}')
print(f"{'variant':16s} {'colbert':>8} {'combined':>9} {'Δceil':>7} {'rescued/missed':>15}")
for v, cbw, cw, d, r, m in rows:
    print(f'{v:16s} {cbw:8.4f} {cw:9.4f} {d:+7.4f} {str(r)+"/"+str(m):>15}  '
          f"{'PASS' if d > 0.01 else 'flat (noise)'}")

In [ ]:
# 9) UNION-MARGINAL gate (the one that matters: ColBERT is 1 of ~7 channels, BM25 already
#    matches raw tags). Does adding the variant's ColBERT raise UNION recall@100, cold vs warm?
#    Compact variants route the compact query to ColBERT only (colbert_compact_query + ctx key).
base_cold, base_warm = rec_mask(cs, 100, cold), rec_mask(cs, 100, warm)
print(f"union baseline recall@100  cold={base_cold:.4f}  warm={base_warm:.4f}\n")
for v in EVAL:
    V = VARIANTS[v]
    ec = {'use_sasrec': True, 'w_sasrec': 1.0, 'use_colbert': True, 'w_colbert': 1.0,
          'colbert_index_name': V['index'], 'colbert_model': model_path(v),
          'colbert_q_len': V['q_len'], 'colbert_compact_query': V['compact_query']}
    ctx_v = [dict(c, colbert_query=queries_compact[i]) if V['compact_query'] else c
             for i, c in enumerate(ctx)]
    u = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                              extra_config=ec)
    cu = u.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx_v)
    print(f"{v:16s} union+colbert recall@100  cold={rec_mask(cu,100,cold):.4f} "
          f"(Δ{rec_mask(cu,100,cold)-base_cold:+.4f})  warm={rec_mask(cu,100,warm):.4f} "
          f"(Δ{rec_mask(cu,100,warm)-base_warm:+.4f})  "
          f"turn1&wall={rec_mask(cu,100,turn1 & is_wall):.4f} (Δ{rec_mask(cu,100,turn1 & is_wall)-cs_w100:+.4f})")

## Section 3.5 — RERANKER two-stage DIAGNOSTIC (where are golds lost?)

Decompose the two-stage on DEV: **pool recall@100 → stage-1 shortlist recall@24 → stage-2 final
recall@20 / nDCG@20**, plus the conditional retention/conversion. Answers the decisive question
behind config 222's flat Blind nDCG: does stage 1 **DROP** golds from the shortlist (→ a stage-1
problem, fixable with flash/bigger-k2), or does stage 2 fail to **CONVERT** golds that ARE in the
shortlist (→ the knowledge ceiling, no reranker tweak helps)? Uses Gemini → run on a SUBSET.

In [ ]:
# 9b) Two-stage reranker decomposition on DEV. ~300 turn-1 queries; Gemini (~$1-3).
import os, numpy as np, pandas as pd
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = os.environ.get('GEMINI_API_KEY') or userdata.get('GEMINI_API_KEY')
except Exception:
    pass
assert os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'), 'Set GEMINI_API_KEY'
from mcrs.rerankers import load_reranker_module
from mcrs.rerankers.two_stage_listwise import compose_two_stage

# rebuild per-query goal/profile aligned with `queries` (faithful to serve), from `dev`
gcat, gspec, prof = [], [], []
for sess in dev:
    df = pd.DataFrame(sess['conversations']); cg = sess.get('conversation_goal') or {}; up = sess.get('user_profile')
    for _ in range(int((df['role'] == 'music').sum())):
        gcat.append(cg.get('category')); gspec.append(cg.get('specificity')); prof.append(up)

N = 300
sub = [i for i, t in enumerate(turn_numbers) if t == 1][:N]
q = [queries[i] for i in sub]; gd = [golds[i] for i in sub]; pool = [cs[i][:100] for i in sub]
kw = dict(user_ids=[user_ids[i] for i in sub], goal_categories=[gcat[i] for i in sub],
          goal_specificities=[gspec[i] for i in sub], user_profiles_raw=[prof[i] for i in sub])

rr = load_reranker_module('llm_listwise_2stage', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
        model_path='gemini-2.5-flash', k=100, k2=24, stage1_model='gemini-2.5-flash-lite',
        rich_candidates=True, thinking_budget=512)
s1 = rr.stage1.rerank(q, pool, topk=100, **kw)              # coarse over 100
shortlist = [o[:rr.k2] for o in s1]
s2 = rr.stage2.rerank(q, shortlist, topk=24, **kw)          # rich over 24
final = compose_two_stage(s1, s2, 20)

# single-stage baseline (215-style: k=50, plain) on the SAME pool, for comparison
ss = load_reranker_module('llm_listwise', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
        model_path='gemini-2.5-flash', k=50, rich_candidates=False, thinking_budget=512)
ss_out = ss.rerank(q, pool, topk=20, **kw)

def rec(c, k):
    return float(np.mean([1.0 if gd[i] in c[i][:k] else 0.0 for i in range(len(gd))]))
def ndcg(c, k):
    v = []
    for i in range(len(gd)):
        ci = c[i][:k]
        v.append(1.0 / np.log2(ci.index(gd[i]) + 2) if gd[i] in ci else 0.0)
    return float(np.mean(v))

print(f'n={len(sub)} turn-1 queries')
print(f'pool recall@100             : {rec(pool,100):.4f}')
print(f'stage-1 shortlist recall@24 : {rec(shortlist,24):.4f}')
print(f'two-stage final recall@20   : {rec(final,20):.4f}   nDCG@20: {ndcg(final,20):.4f}')
print(f'single-stage   recall@20    : {rec(ss_out,20):.4f}   nDCG@20: {ndcg(ss_out,20):.4f}')
inp = [i for i in range(len(gd)) if gd[i] in pool[i][:100]]
ins = [i for i in inp if gd[i] in shortlist[i]]
intop = [i for i in ins if gd[i] in final[i][:20]]
print(f'\nDECOMPOSITION (of {len(inp)} golds present in the pool):')
print(f'  stage-1 RETENTION  (pool->shortlist@24): {len(ins)}/{len(inp)} = {len(ins)/max(len(inp),1):.1%}')
print(f'  stage-2 CONVERSION (shortlist->top20)  : {len(intop)}/{len(ins)} = {len(intop)/max(len(ins),1):.1%}')
print('READ: low retention -> stage 1 DROPS golds (fixable: flash/bigger k2). High retention but low')
print('      conversion -> stage 2 cannot rank known-in-shortlist golds = the knowledge ceiling.')

## Section 4 — Pre-registered decision rules

Per the campaign's hard-won lessons (internal val is a TRAP; only the recall WALL clears the
80-session ±0.05 noise; doc/query additions inflate internal metrics while dev stays flat —
EXP-016, CLAP):

- **PASS → promote to a Blind config** iff **union** recall@100 (cell 9) rises **beyond noise**
  on the wall/cold segment **AND** the wall-rescue gate (cell 8) shows ColBERT surfacing golds
  the union missed. (EXP-217's incremental value must be *semantic* tag matching beyond BM25's
  existing exact tag match — if it just duplicates BM25, the union won't move.)
- **Standalone ↑ but union flat → SHELVE** (redundant in the ensemble — the EXP-016 pattern).
- **Any axis ↓ → REJECT.**
- nDCG guard: if recall↑ but nDCG↓ (run the #82-p3-ndcg conversion A/B), the addition is noise
  (MaxSim score-inflation). Do NOT gate on internal/standalone val.